In [ ]:
import os
import pandas as pd
from src import config, ingestion
from pathlib import Path

In [ ]:
os.makedirs(config.DATA_RAW_DIR, exist_ok=True)

In [ ]:
historical_weather = ingestion.fetch_all_zones(
    start_date=config.HISTORICAL_START,
    end_date=config.HISTORICAL_END,
    mode="historical"
)

In [ ]:
for zone, df in historical_weather.items():
    df.to_csv(config.DATA_RAW_DIR / f"{zone.lower()}_weather_historical.csv", index=False)

In [ ]:
historical_flood = ingestion.fetch_all_zones(
    start_date=config.HISTORICAL_START,
    end_date=config.HISTORICAL_END,
    mode="flood"
)

In [ ]:
for zone, df in historical_flood.items():
    if not df.empty:
        df.to_csv(config.DATA_RAW_DIR / f"{zone.lower()}_flood_historical.csv", index=False)

In [ ]:
forecast_weather = ingestion.fetch_all_zones(
    start_date=None, 
    end_date=None, 
    mode="forecast"
)

In [ ]:
for zone, df in forecast_weather.items():
    df.to_csv(config.DATA_RAW_DIR / f"{zone.lower()}_weather_forecast.csv", index=False)

In [ ]:
def audit_data(df, name):
    expected = pd.date_range(start=df['time'].min(), end=df['time'].max())
    gaps = expected.difference(df['time'])
    nulls = df.isnull().sum()
    return {
        "Dataset": name,
        "Rows": len(df),
        "Start": df['time'].min(),
        "End": df['time'].max(),
        "Gaps": len(gaps),
        "Nulls": nulls[nulls > 0].to_dict()
    }

In [ ]:
results = []
for z, df in historical_weather.items():
    results.append(audit_data(df, f"{z} Weather"))
for z, df in historical_flood.items():
    if not df.empty:
        results.append(audit_data(df, f"{z} Flood"))

pd.DataFrame(results)